# Underwater ResNet Restoration Colab Workflow

Use this notebook after uploading `underwater_resnet_project` to `MyDrive`. Datasets, checkpoints, logs, and results stay on Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/bedirhanozturk1/Underwater_Resnet_Restoration.git'
DRIVE_ROOT = '/content/drive/MyDrive/underwater_resnet_project'
!test -d /content/Underwater_Resnet_Restoration || git clone {REPO_URL} /content/Underwater_Resnet_Restoration
%cd /content/Underwater_Resnet_Restoration
!git pull
!pip install -r requirements.txt

In [ ]:
import os
required = [
    f'{DRIVE_ROOT}/datasets/clear_underwater_color_patch/canon_patch',
    f'{DRIVE_ROOT}/datasets/turbidty_underwater_color_patch',
    f'{DRIVE_ROOT}/splits/train.txt',
    f'{DRIVE_ROOT}/splits/val.txt',
    f'{DRIVE_ROOT}/splits/test.txt',
]
for path in required:
    print(path, 'OK' if os.path.exists(path) else 'MISSING')

## Train Baseline
Run this first for a fair baseline. Increase `--epochs` if Colab time allows.

In [ ]:
!python scripts/train_model.py \
  --model baseline \
  --epochs 50 \
  --batch-size 16 \
  --clear-dir {DRIVE_ROOT}/datasets/clear_underwater_color_patch/canon_patch \
  --turbid-dir {DRIVE_ROOT}/datasets/turbidty_underwater_color_patch \
  --split-dir {DRIVE_ROOT}/splits \
  --checkpoint-dir {DRIVE_ROOT}/checkpoints \
  --log-dir {DRIVE_ROOT}/logs

## Train Residual Backbone
This is the proposed ResNet-style residual denoising backbone.

In [ ]:
!python scripts/train_model.py \
  --model residual \
  --epochs 50 \
  --batch-size 16 \
  --clear-dir {DRIVE_ROOT}/datasets/clear_underwater_color_patch/canon_patch \
  --turbid-dir {DRIVE_ROOT}/datasets/turbidty_underwater_color_patch \
  --split-dir {DRIVE_ROOT}/splits \
  --checkpoint-dir {DRIVE_ROOT}/checkpoints \
  --log-dir {DRIVE_ROOT}/logs

## Resume After Disconnect
Change `baseline` to `residual` if resuming the proposed model.

In [ ]:
!python scripts/train_model.py \
  --model residual \
  --epochs 50 \
  --batch-size 16 \
  --resume {DRIVE_ROOT}/checkpoints/residual/latest.pth \
  --clear-dir {DRIVE_ROOT}/datasets/clear_underwater_color_patch/canon_patch \
  --turbid-dir {DRIVE_ROOT}/datasets/turbidty_underwater_color_patch \
  --split-dir {DRIVE_ROOT}/splits \
  --checkpoint-dir {DRIVE_ROOT}/checkpoints \
  --log-dir {DRIVE_ROOT}/logs

## Evaluate And Run Inference

In [ ]:
!python scripts/evaluate_model.py \
  --checkpoint {DRIVE_ROOT}/checkpoints/residual/best.pth \
  --clear-dir {DRIVE_ROOT}/datasets/clear_underwater_color_patch/canon_patch \
  --turbid-dir {DRIVE_ROOT}/datasets/turbidty_underwater_color_patch \
  --split-file {DRIVE_ROOT}/splits/test.txt \
  --result-dir {DRIVE_ROOT}/results/evaluation

In [ ]:
!python scripts/run_inference.py \
  --checkpoint {DRIVE_ROOT}/checkpoints/residual/best.pth \
  --input-dir {DRIVE_ROOT}/datasets/auxiliary_unpaired/turbidty_from_video_frame \
  --output-dir {DRIVE_ROOT}/results/inference/residual_auxiliary_frames \
  --limit 20